# Device + Crafter smoke check

Quick env sanity check. For **real functionality + graphs**, use:

- `01_perception_reconstructions.ipynb` — load AE, reconstruct, metric charts
- `02_rssm_live_diagnostics.ipynb` — run RSSM observe/imagine, plot mechanism graphs inline

**Expect here:** CUDA available (RTX 5080), a `(64, 64, 3)` Crafter frame, and tracked M1/M2 PNGs when present.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display

from training.device import configure_runtime, describe_device, get_device, warn_if_not_cuda

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

print(f"cwd: {Path.cwd()}")
print(f"torch {torch.__version__}")
print(f"CUDA built: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = get_device()
configure_runtime(device)
print(f"using device: {describe_device(device)}")
warn_if_not_cuda(device)


## Crafter frame

Reset once and plot a single observation. Confirms the Gymnasium wrapper + render path without collecting a full episode.


In [ ]:
import gymnasium as gym
import envs  # noqa: F401 — registers CrafterReward-v1

env = gym.make("CrafterReward-v1")
obs, info = env.reset(seed=0)
env.close()

print(f"obs shape: {getattr(obs, 'shape', type(obs))}")
print(f"obs dtype: {getattr(obs, 'dtype', type(obs))}")

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(obs)
ax.set_title("CrafterReward-v1 reset")
ax.axis("off")
plt.show()


## Tracked milestone visuals

These PNGs are committed under `results/` as milestone proof. Re-open them here when you want a glance without digging through Finder or TensorBoard.


In [ ]:
paths = [
    ROOT / "results/m1/recon_final.png",
    ROOT / "results/m1/recon_diverse_check.png",
    ROOT / "results/m2/latent_entropy.png",
    ROOT / "results/m2/imagination_divergence.png",
    ROOT / "results/m2/h_trajectory_pca.png",
]

for path in paths:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        continue
    print(path.relative_to(ROOT))
    display(Image(filename=str(path)))
